In [1]:
import pandas as pd

# Define column names matching your -outfmt 6 string
columns = [
    "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore",
    "qlen", "slen", "qcovs", "qcovhsp", "nident", "positive",
    "gaps", "ppos", "score"
]

# Read the tabular BLAST output
df = pd.read_csv("blast_results.tsv", sep="\t", header=None, names=columns)

# Preview
df.head()

,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,...,bitscore,qlen,slen,qcovs,qcovhsp,nident,positive,gaps,ppos,score
0,SRR28250234|k119_1252/Avihepatovirus/988/717-7...,IMGVR_UViG_3300045669_000203|3300045669|Ga0490...,79.585,965,192,5,23,986,9471,8511,...,686.0,988,16967,98,98,768,768,5,79.59,371
1,SRR28250234|k119_1252/Avihepatovirus/988/717-7...,IMGVR_UViG_2660238363_000002|2660238363|266030...,79.209,986,203,2,2,986,38389,39373,...,684.0,988,41931,99,99,781,781,2,79.21,370
2,SRR28250234|k119_1252/Avihepatovirus/988/717-7...,IMGVR_UViG_3300045633_000171|3300045633|Ga0486...,79.209,986,201,4,2,986,98247,99229,...,682.0,988,113099,99,99,781,781,4,79.21,369
3,SRR28250234|k119_1252/Avihepatovirus/988/717-7...,IMGVR_UViG_3300045618_000180|3300045618|Ga0486...,79.482,965,195,3,23,986,5617,4655,...,682.0,988,114360,98,98,767,767,3,79.48,369
4,SRR28250234|k119_1252/Avihepatovirus/988/717-7...,IMGVR_UViG_3300045647_000138|3300045647|Ga0486...,79.209,986,198,7,2,986,7468,8447,...,678.0,988,35467,99,99,781,781,7,79.21,367


In [5]:
# Case-insensitive match on "astro" anywhere in qseqid
astro_df = df[df["qseqid"].str.contains("astro", case=False, na=False)]

# Keep only the best hit per query (highest bitscore; ties broken by lowest evalue)
astro_best = (
    astro_df
    .sort_values(["qseqid", "bitscore", "evalue"], ascending=[True, False, True])
    .drop_duplicates(subset="qseqid", keep="first")
)

astro_best.head()

astro_best.to_csv('our_astroviruses_blasted_on_metavr.csv')

In [9]:
import re

# --- 1. Load the new table (adjust filename/sep as needed) ---
# If it's a tab-separated file:
meta = pd.read_csv("complete_genomes_3orfs_min5000.csv", sep='\t', dtype=str)

# --- 2. Extract contig IDs like k141_382090 from "Contig_ID: k141_382090 | BioProject: ..." ---
def extract_contig(s):
    m = re.search(r"Contig_ID:\s*(\S+)", str(s))
    return m.group(1) if m else None

meta["contig_id"] = meta["Инфо"].apply(extract_contig)

# Drop rows where extraction failed
meta_clean = meta.dropna(subset=["contig_id"]).copy()
print("Contigs found in metadata table:", len(meta_clean))

# --- 3. Extract contig ID from qseqid like:
#     SRR27962826|k141_673/Astroviridae/22365/14298-13341/.../1B
#     -> k141_673
def extract_qseqid_contig(q):
    # split on '|' first, take part after it, then split on '/'
    parts = str(q).split("|", 1)
    if len(parts) < 2:
        return None
    return parts[1].split("/", 1)[0]

astro_best["qseqid_contig"] = astro_best["qseqid"].apply(extract_qseqid_contig)

# --- 4. Keep only blast rows whose contig is in the metadata table ---
target_contigs = set(meta_clean["contig_id"])
filtered = astro_best[astro_best["qseqid_contig"].isin(target_contigs)].copy()
print("Blast hits matching metadata contigs:", len(filtered))

# --- 5. Attach "Бласт хит" column from metadata ---
#    Merge on contig_id == qseqid_contig
meta_slim = meta_clean[["contig_id", "Бласт хит"]].rename(
    columns={"contig_id": "qseqid_contig", "Бласт хит": "blast_hit_meta"}
)
result = filtered.merge(meta_slim, on="qseqid_contig", how="left")

# Preview
result[["qseqid", "qseqid_contig", "sseqid", "pident", "evalue", "bitscore", "blast_hit_meta"]].head()

# Save
result.to_csv("blast_hits_matched_to_metadata.tsv", sep="\t", index=False)

Contigs found in metadata table: 36
Blast hits matching metadata contigs: 25
